# Module-09: Guided Lab

In [ ]:
# Install NLTK (Natural Language Toolkit) for text processing
!pip install nltk

# Install Sumy for automatic text summarization
!pip install sumy

# Install Gensim for topic modeling and word embeddings
!pip install gensim

# Install Hugging Face Transformers for advanced NLP models
!pip install transformers

This code demonstrates how to Implementing Extractive Summarization and a simple extractive text summarizer using Python and NLTK. Extractive summarization means the program selects the most important original sentences—without rewriting them—to create a shorter version of the text. The process includes splitting the text into sentences, cleaning each sentence by removing stop words and punctuation, counting how frequently each meaningful word appears, scoring sentences based on these word frequencies, and finally choosing the top-scoring sentences to form a concise summary. This is an easy and intuitive way to introduce students to automatic summarization techniques.

In [ ]:
# ----------------------------------------------------------
# SIMPLE TEXT SUMMARIZATION WITH NLTK (TERM FREQUENCY)
# ----------------------------------------------------------
# In this example, we:
# 1) Break a paragraph into individual sentences.
# 2) Clean each sentence (lowercase, remove stop words and punctuation).
# 3) Count how often each word appears (term frequency).
# 4) Give each sentence a score based on the words it contains.
# 5) Pick the top sentences to form a short summary.
# ----------------------------------------------------------

# NLTK (Natural Language Toolkit) is a popular library for working with text.
import nltk

# sent_tokenize  -> splits text into sentences.
# word_tokenize  -> splits a sentence into individual words.
from nltk.tokenize import sent_tokenize, word_tokenize

# stopwords -> list of common words like "the", "and", "is" that often
#              do not add much meaning.
from nltk.corpus import stopwords

# FreqDist -> helps us count how frequently each word appears.
from nltk.probability import FreqDist

# cosine_distance, numpy, and networkx are more advanced tools often used
# in graph-based summarization methods (like TextRank).
# We import them here for completeness, but they are not used in this simple example.
from nltk.cluster.util import cosine_distance
import numpy as np
import networkx as nx

# ----------------------------------------------------------
# DOWNLOAD NLTK RESOURCES (RUN ONCE PER ENVIRONMENT)
# ----------------------------------------------------------
# 'punkt'     -> data needed for sentence and word tokenization.
# 'stopwords' -> list of common words in many languages.
# 'punkt_tab' -> extra tokenizer support (needed in newer NLTK versions).
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

# ----------------------------------------------------------
# SAMPLE TEXT TO SUMMARIZE
# ----------------------------------------------------------
# This is the paragraph we want to summarize.
text = """Natural language processing (NLP) is a subfield of linguistics, computer science, and artificial intelligence
concerned with the interactions between computers and human language, in particular how to program computers to process
and analyze large amounts of natural language data. Challenges in natural language processing frequently involve speech
recognition, natural language understanding, and natural language generation."""

# ----------------------------------------------------------
# 1. SPLIT TEXT INTO SENTENCES
# ----------------------------------------------------------
# sent_tokenize() takes a long text and returns a list of sentences.
sentences = sent_tokenize(text)

# Get a set of English stop words, such as "the", "is", "and", etc.
stop_words = set(stopwords.words('english'))

# ----------------------------------------------------------
# 2. FUNCTION TO CLEAN A SENTENCE
# ----------------------------------------------------------
def preprocess_sentence(sentence):
    """
    Convert a sentence into a cleaned list of words:
    - lowercase the sentence
    - split into words
    - keep only alphanumeric words (no punctuation)
    - remove stop words
    """
    # Convert sentence to lowercase and split into words.
    words = word_tokenize(sentence.lower())

    # Keep only words that:
    # 1) are alphanumeric (letters/numbers only)
    # 2) are not in the stop word list
    words = [word for word in words if word.isalnum() and word not in stop_words]
    return words

# ----------------------------------------------------------
# 3. SCORE SENTENCES BASED ON WORD FREQUENCY
# ----------------------------------------------------------
def score_sentences(sentences):
    """
    Give each sentence a score based on the frequency of its words.
    Sentences that contain many frequent (important) words get higher scores.
    """
    sentence_scores = []

    # Flatten all words from all sentences into one big list and build a frequency distribution.
    all_words = [word for sentence in sentences for word in preprocess_sentence(sentence)]
    word_frequencies = FreqDist(all_words)

    # Now, score each individual sentence.
    for sentence in sentences:
        # Preprocess the sentence (get cleaned words).
        words = preprocess_sentence(sentence)

        # Sentence score = sum of the frequencies of its words.
        sentence_score = sum(word_frequencies[word] for word in words)

        # Store the sentence together with its score.
        sentence_scores.append((sentence, sentence_score))

    return sentence_scores

# ----------------------------------------------------------
# 4. PICK THE TOP SCORING SENTENCES
# ----------------------------------------------------------
def select_sentences(sentence_scores, num_sentences=2):
    """
    Sort sentences by their score (from highest to lowest)
    and pick the top 'num_sentences' of them.
    """
    # Sort list of (sentence, score) pairs by score in descending order.
    sentence_scores.sort(key=lambda x: x[1], reverse=True)

    # Extract only the sentence text for the top N sentences.
    selected_sentences = [sentence[0] for sentence in sentence_scores[:num_sentences]]
    return selected_sentences

# ----------------------------------------------------------
# 5. RUN THE SUMMARIZATION PIPELINE
# ----------------------------------------------------------
# Step 1: Score each sentence.
sentence_scores = score_sentences(sentences)

# Step 2: Select the best sentences for our summary.
summary_sentences = select_sentences(sentence_scores, num_sentences=2)

# Step 3: Join the selected sentences back into a single string.
summary = ' '.join(summary_sentences)

# ----------------------------------------------------------
# PRINT THE SUMMARY
# ----------------------------------------------------------
print("Summary:")
print(summary)


This code demonstrates a more advanced approach to extractive summarization by using TF-IDF to convert sentences into numerical vectors, computing similarity between sentences, and applying the TextRank algorithm (a graph-based ranking method) to select the most important sentences. Unlike simple frequency-based summarization, this technique evaluates how sentences relate to each other, producing higher-quality summaries that better capture the main ideas of the text.

In [ ]:
# ----------------------------------------------------------
# TEXT SUMMARIZATION WITH TF-IDF + TEXTRANK (BEGINNER FRIENDLY)
# ----------------------------------------------------------
# In this example, we:
# 1) Break a long paragraph into sentences.
# 2) Clean each sentence (lowercase, remove stop words, remove punctuation).
# 3) Turn each sentence into a TF-IDF vector (a list of numbers).
# 4) Measure how similar sentences are to each other (cosine similarity).
# 5) Build a graph of sentences and run PageRank (TextRank).
# 6) Pick the top-ranked sentences to form a summary.
# ----------------------------------------------------------

# NLTK is a popular Python library for working with human language (text).
import nltk

# sent_tokenize: split long text into sentences.
# word_tokenize: split sentences into individual words.
from nltk.tokenize import sent_tokenize, word_tokenize

# stopwords: a list of common words like "the", "is", "and" that we often remove.
from nltk.corpus import stopwords

# NumPy is a library for working with arrays and numerical operations.
import numpy as np

# NetworkX is a library for working with graphs (nodes and edges).
import networkx as nx

# TfidfVectorizer converts text into TF-IDF vectors (numbers).
from sklearn.feature_extraction.text import TfidfVectorizer

# cosine_similarity measures how similar two vectors (sentences) are.
from sklearn.metrics.pairwise import cosine_similarity

# ----------------------------------------------------------
# DOWNLOAD NLTK RESOURCES (RUN ONCE PER ENVIRONMENT)
# ----------------------------------------------------------
# 'punkt'     -> required for sentence and word tokenization.
# 'stopwords' -> list of common words (like "the", "is", "and") for many languages.
nltk.download('punkt')
nltk.download('stopwords')

# ----------------------------------------------------------
# SAMPLE TEXT TO SUMMARIZE
# ----------------------------------------------------------
# This is the text we want to summarize.
# You can replace this with any other long paragraph or article.
text = """Natural language processing (NLP) is a subfield of linguistics, computer science, and artificial intelligence
concerned with the interactions between computers and human language, in particular how to program computers to process
and analyze large amounts of natural language data. Challenges in natural language processing frequently involve speech
recognition, natural language understanding, and natural language generation."""

# ----------------------------------------------------------
# 1) SPLIT TEXT INTO SENTENCES AND BASIC CLEANUP
# ----------------------------------------------------------
# sent_tokenize() splits the text into a list of sentences.
# We also remove any empty sentences just in case.
sentences = [s for s in sent_tokenize(text) if s.strip()]

# Load a set of English stop words (like "the", "is", "and").
stop_words = set(stopwords.words('english'))

def normalize(s):
    """
    Clean (normalize) a sentence:
    - convert to lowercase (so 'NLP' and 'nlp' are treated the same)
    - split into words (tokens)
    - keep only alphanumeric words (remove punctuation)
    - remove stop words (common words that add little meaning)
    - join the cleaned words back into a single string
    """
    # Convert to lowercase and split into individual words.
    tokens = word_tokenize(s.lower())

    # Keep words that are:
    # - alphanumeric (letters and/or numbers)
    # - NOT in the stop word list
    tokens = [w for w in tokens if w.isalnum() and w not in stop_words]

    # Join the cleaned words back into one string, separated by spaces.
    return " ".join(tokens)

# Apply normalization to every sentence.
normalized = [normalize(s) for s in sentences]

# ----------------------------------------------------------
# 2) CONVERT SENTENCES INTO TF-IDF VECTORS
# ----------------------------------------------------------
# TfidfVectorizer:
# - learns the vocabulary from the text
# - computes TF-IDF scores for each word in each sentence
vectorizer = TfidfVectorizer()

# fit_transform():
# - fit: learn the vocabulary from "normalized" sentences
# - transform: create a matrix of TF-IDF features
#
# X has shape (number_of_sentences, vocabulary_size).
X = vectorizer.fit_transform(normalized)

# ----------------------------------------------------------
# 3) COMPUTE COSINE SIMILARITY BETWEEN SENTENCES
# ----------------------------------------------------------
# cosine_similarity(X, X) creates a matrix where:
# - each row and column represent a sentence
# - each value is how similar two sentences are (between 0 and 1)
sim_matrix = cosine_similarity(X, X)

# We set the diagonal to 0 so a sentence is not "similar to itself"
# with a dominating score. This helps in building the graph.
np.fill_diagonal(sim_matrix, 0.0)

# ----------------------------------------------------------
# 4) TEXTRANK USING NETWORKX (PAGERANK ON THE SIMILARITY GRAPH)
# ----------------------------------------------------------
# We think of each sentence as a node in a graph.
# If two sentences are similar, we connect them with an edge.
# The weight of the edge is the similarity score.
graph = nx.from_numpy_array(sim_matrix)

# PageRank (used in TextRank) gives each sentence a "score"
# based on how important it is in the graph.
scores = nx.pagerank(graph)

# ----------------------------------------------------------
# 5) SELECT THE TOP-RANKED SENTENCES AS SUMMARY
# ----------------------------------------------------------
# Decide how many sentences we want in the summary.
num_sentences = 2

# scores is a dictionary: {sentence_index: score}
# We pair each sentence score with the original sentence text.
ranked = sorted(
    ((scores[i], s) for i, s in enumerate(sentences)),
    reverse=True  # highest score first
)

# Pick the top "num_sentences" sentences.
summary_sentences = [s for _, s in ranked[:num_sentences]]

# Join the selected sentences together to form the final summary.
summary = " ".join(summary_sentences)

# ----------------------------------------------------------
# PRINT THE SUMMARY
# ----------------------------------------------------------
print("Summary:")
print(summary)


This code demonstrates how to convert text into numerical form using a term–document matrix and then apply Truncated SVD (a simplified form of Latent Semantic Analysis) to reduce the data into a lower-dimensional “topic space.” It shows how documents and individual terms can be represented as compact vectors that reveal underlying patterns or topics in the text.

In [ ]:
# ----------------------------------------------------------
# Simple Example: Turning Text into Numbers with SVD (LSA Idea)
# ----------------------------------------------------------
# In this example, we:
# 1) Start with a few short text documents (sentences).
# 2) Turn the text into a term–document matrix (words vs. documents).
# 3) Use TruncatedSVD to reduce the dimensionality (similar idea to LSA).
# 4) Look at the new, lower-dimensional representations of documents and terms.
# ----------------------------------------------------------

# ----------------------------------------------------------
# 1. SAMPLE TEXT DOCUMENTS
# ----------------------------------------------------------
# Each string is one "document" (here, just a short sentence).
documents = [
    "The cat sat on the mat",
    "The dog chased the ball",
    "The bird flew in the sky"
]

# ----------------------------------------------------------
# 2. CREATE A TERM–DOCUMENT MATRIX
# ----------------------------------------------------------
# CountVectorizer:
# - builds a vocabulary of all unique words in the documents
# - creates a matrix where:
#     rows    = documents
#     columns = words (terms)
#     values  = how many times each word appears in each document
from sklearn.feature_extraction.text import CountVectorizer

# Create the vectorizer
vectorizer = CountVectorizer()

# Learn the vocabulary and build the term–document matrix.
# term_document_matrix is a sparse matrix of shape (n_documents, n_terms).
term_document_matrix = vectorizer.fit_transform(documents)

# ----------------------------------------------------------
# 3. REDUCE DIMENSIONS WITH TRUNCATED SVD
# ----------------------------------------------------------
# TruncatedSVD:
# - is a version of Singular Value Decomposition (SVD) that works on
#   large, sparse matrices.
# - is often used in Latent Semantic Analysis (LSA) to find hidden
#   "topics" in text.
from sklearn.decomposition import TruncatedSVD

# n_components = 2 means we want to reduce the data down to 2 dimensions.
# (This is useful for visualization or for seeing basic topic structure.)
svd = TruncatedSVD(n_components=2)

# Fit SVD on the term–document matrix and transform it.
# lsa_matrix has shape (n_documents, 2), so each document is now a
# 2-dimensional vector (a point in topic space).
lsa_matrix = svd.fit_transform(term_document_matrix)

# ----------------------------------------------------------
# 4. EXTRACT TERMS AND TOPIC INFORMATION
# ----------------------------------------------------------
# Get the list of all terms (words) that CountVectorizer learned.
terms = vectorizer.get_feature_names_out()

# svd.components_ is a matrix of shape (n_components, n_terms).
# Each row represents one "topic" and each column represents a term.
# The values tell us how strongly each term is related to each topic.
lsa_topics = svd.components_

# ----------------------------------------------------------
# 5. PRINT THE RESULTS
# ----------------------------------------------------------
print("Terms (vocabulary):")
print(terms)

print("\nReduced Document Representations (each row = a document, each column = a topic):")
print(lsa_matrix)

print("\nReduced Term Representations (each row = a topic, each column = a term):")
print(lsa_topics)


This code demonstrates how to use a pre-trained BART transformer model to perform abstractive text summarization. It loads the BART model and tokenizer, converts the input text into tokens the model can process, generates a concise summary using beam search, and then decodes the output back into human-readable text.

In [ ]:
# ----------------------------------------------------------
# TEXT SUMMARIZATION WITH BART (TRANSFORMERS)
# ----------------------------------------------------------
# In this example, we use a pre-trained BART model to automatically
# create a short summary of a longer text.
#
# Steps:
# 1) Load a pre-trained BART model and its tokenizer.
# 2) Turn the input text into tokens (numbers) the model understands.
# 3) Ask the model to generate a summary.
# 4) Decode the model output back into readable text.
# ----------------------------------------------------------

# Import the BART model and tokenizer from the transformers library.
# BartForConditionalGeneration  -> the model that generates text (such as summaries).
# BartTokenizer                 -> converts between text and token IDs (numbers).
from transformers import BartForConditionalGeneration, BartTokenizer

# ----------------------------------------------------------
# 1. LOAD THE PRE TRAINED BART MODEL AND TOKENIZER
# ----------------------------------------------------------
# "facebook/bart-large-cnn" is a BART model already fine tuned for summarization.
model_name = "facebook/bart-large-cnn"

# Load the model weights and architecture from the internet or cache.
model = BartForConditionalGeneration.from_pretrained(model_name)

# Load the matching tokenizer so it uses the same vocabulary as the model.
tokenizer = BartTokenizer.from_pretrained(model_name)

# ----------------------------------------------------------
# 2. SAMPLE TEXT TO SUMMARIZE
# ----------------------------------------------------------
# This is the long text we want to summarize.
# You can replace this with any article or paragraph.
text = """Natural language processing (NLP) is a subfield of linguistics, computer science, and artificial intelligence
concerned with the interactions between computers and human language, in particular how to program computers to process
and analyze large amounts of natural language data. Challenges in natural language processing frequently involve speech
recognition, natural language understanding, and natural language generation."""

# ----------------------------------------------------------
# 3. TOKENIZE AND ENCODE THE INPUT TEXT
# ----------------------------------------------------------
# We add the word "summarize:" at the beginning.
# This is a common style for some models that supports instruction like input.
#
# tokenizer.encode:
#   - turns the text into a list of token IDs (numbers)
#   - return_tensors="pt" gives us a PyTorch tensor as output
#   - max_length=512 limits the input length
#   - truncation=True cuts off text that is too long
inputs = tokenizer.encode(
    "summarize: " + text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# ----------------------------------------------------------
# 4. GENERATE THE SUMMARY
# ----------------------------------------------------------
# model.generate creates the summary based on the input tokens.
#
# Parameters:
#   max_length    -> maximum length of the summary in tokens.
#   min_length    -> minimum length of the summary in tokens.
#   length_penalty -> value > 1.0 makes shorter summaries more likely.
#   num_beams     -> number of beams for beam search (higher is usually better but slower).
#   early_stopping -> stop when all beams have reached the end of the summary.
summary_ids = model.generate(
    inputs,
    max_length=150,
    min_length=40,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

# Decode the token IDs back into a regular string.
# skip_special_tokens=True removes tokens like <s> and </s>.
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# ----------------------------------------------------------
# 5. PRINT THE SUMMARY
# ----------------------------------------------------------
print("Summary:")
print(summary)


Advanced Abstractive Summarization Techniques
This code demonstrates how to use Advanced Abstractive Summarization Techniques using  pre-trained T5 transformer model to perform abstractive text summarization. The T5 tokenizer converts the input text into tokens, the model generates a shorter rewritten version using beam search, and the output is then decoded back into readable text. This provides a simple and effective way to create high-quality summaries using modern AI models.



In [ ]:
# ----------------------------------------------------------
# TEXT SUMMARIZATION WITH T5 (TRANSFORMERS)
# ----------------------------------------------------------
# In this example, we use a pre-trained T5 model to automatically
# generate a short summary of a longer piece of text.
#
# Main steps:
# 1) Load a pre-trained T5 model and its tokenizer.
# 2) Turn the input text into tokens (numbers) the model understands.
# 3) Ask the model to generate a summary.
# 4) Convert the model output back into readable text.
# ----------------------------------------------------------

# Import the T5 model and tokenizer from the transformers library.
# T5ForConditionalGeneration -> the model that generates text (like a summary).
# T5Tokenizer                -> converts between text and token IDs (numbers).
from transformers import T5ForConditionalGeneration, T5Tokenizer

# ----------------------------------------------------------
# 1. LOAD THE PRE-TRAINED T5 MODEL AND TOKENIZER
# ----------------------------------------------------------
# "t5-small" is a smaller version of T5 that is faster and lighter,
# which makes it good for learning and experimenting.
model_name = "t5-small"

# Load the T5 model (this may download weights the first time you run it).
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Load the matching tokenizer so it uses the same vocabulary as the model.
tokenizer = T5Tokenizer.from_pretrained(model_name)

# ----------------------------------------------------------
# 2. SAMPLE TEXT TO SUMMARIZE
# ----------------------------------------------------------
# This is the long text we want the model to summarize.
# You can replace this text with any article, paragraph, or document.
text = """Natural language processing (NLP) is a subfield of linguistics, computer science, and artificial intelligence
concerned with the interactions between computers and human language, in particular how to program computers to process
and analyze large amounts of natural language data. Challenges in natural language processing frequently involve speech
recognition, natural language understanding, and natural language generation."""

# ----------------------------------------------------------
# 3. TOKENIZE AND ENCODE THE INPUT TEXT
# ----------------------------------------------------------
# T5 expects the task to be written in the input, such as "summarize: <text>".
# Here we prepend "summarize: " to tell the model that we want a summary.
#
# tokenizer.encode:
#   - turns the text into a list of token IDs (numbers)
#   - return_tensors="pt" gives a PyTorch tensor (needed for the model)
#   - max_length=512 limits how long the input can be
#   - truncation=True cuts off extra text if it’s too long
inputs = tokenizer.encode(
    "summarize: " + text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# ----------------------------------------------------------
# 4. GENERATE THE SUMMARY
# ----------------------------------------------------------
# model.generate creates the summary based on the input tokens.
#
# Important parameters:
#   max_length     -> maximum length of the summary (in tokens).
#   min_length     -> minimum length of the summary (in tokens).
#   length_penalty -> value > 1.0 encourages shorter summaries.
#   num_beams      -> beam search size (more beams = better quality but slower).
#   early_stopping -> stop when all beams have finished generating.
summary_ids = model.generate(
    inputs,
    max_length=150,
    min_length=40,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

# Decode the summary tokens back into plain text.
# skip_special_tokens=True removes tokens like <pad> and </s>.
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# ----------------------------------------------------------
# 5. PRINT THE SUMMARY
# ----------------------------------------------------------
print("Summary:")
print(summary)
